In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if that is your actual folder name.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OCQ_I.xpt"
csv_file = DATA_DIR / "OCQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 OCQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OCD150",
    "OCQ180",
    "OCQ210",
    "OCQ260",
    "OCD270",
    "OCQ600",
    "OCQ610",
    "OCQ630",
    "OCQ640",
    "OCQ660",
    "OCQ380",
    "OCD390G",
    "OCD395"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "OCQ_I columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6327:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 14:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,327 rows x 14 columns."
)


# ============================================================
# 4. DEFINE DECIMAL / INTEGER FIELDS
# ============================================================

#
# Direct inspection of attached OCQ_I:
#
#     genuine decimal variables = NONE
#     string variables          = NONE
#
#
# All 14 released variables contain only:
#
#     IDs
#     categories
#     hours
#     days
#     months
#     special whole-number codes
#
#
# Therefore all 14 can ultimately become nullable Int64
# AFTER the validated tiny-zero repairs.
#
#
# IMPORTANT:
#
# This is specific to OCQ_I.
#
# It does NOT change the broader NHANES fidelity rule:
#
#     Dietary genuine decimals
#     Laboratory continuous values
#     Examination measurements
#     survey/sample weights
#
# remain exactly AS IS.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 14:

    raise ValueError(
        "Expected exactly 14 whole-number/code fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# NEVER blanket-replace this value before validating
# the exact variables and counts.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 14 OCQ_I variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached OCQ_I.xpt:
#
#     OCD270      105
#     OCQ610    2,495
#     OCQ640    2,918
#     OCD395       19
#
# TOTAL        5,537
#
#
# CDC supports legitimate zero values:
#
# OCD270:
#     0 = Less than 1 month
#
# OCQ610:
#     0-30 days allowed
#
# OCQ640:
#     0-30 days allowed
#
# OCD395:
#     0 = Less than 1 month
#

EXPECTED_TINY_COUNTS = {
    "OCD270": 105,
    "OCQ610": 2495,
    "OCQ640": 2918,
    "OCD395": 19
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated OCQ_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 5537:

    raise ValueError(
        f"Unexpected tiny-value total: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 4:

    raise ValueError(
        "Expected tiny values in exactly 4 variables."
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before tiny-value correction."
    )


print(
    "\nPASS: Exact 5,537-value tiny-zero pattern "
    "across 4 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 5537:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df becomes our strict numerical
# fidelity reference.
#
# ONLY:
#
#     validated tiny -> 0
#
# modifications are allowed.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() is used ONLY as a diagnostic test.
#
# NO dataframe values are changed.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = fractional_count


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal variables "
        "found in OCQ_I.\n\n"
        "These will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "\nPASS: No genuine decimal variables "
    "found in OCQ_I."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 12. VALIDATE ALL VARIABLES ARE WHOLE-NUMBER
# ============================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All 14 OCQ_I variables contain "
    "only whole-number values/codes."
)


# ============================================================
# 13. CONVERT ALL FIELDS TO NULLABLE Int64
# ============================================================

#
# NO .round() during actual conversion.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 14 OCQ_I variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 14. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 15. CDC CHECK - OCD150
# ============================================================

#
# Work status last week.
#

check_frequency(
    "OCD150",
    {
        1: 3386,
        2: 133,
        3: 259,
        4: 2539,
        7: 1,
        9: 7
    },
    2
)


# ============================================================
# 16. CDC CHECK - OCQ180
# ============================================================

#
# Hours worked last week.
#
# CDC 2015-16 top-code:
#
#     80 = 80 hours or more
#
#     1-79   = 3,326
#     80     =    58
#     77777  =     1
#     99999  =     1
#
# Missing = 2,941
#

ocq180_range = int(
    df["OCQ180"]
    .between(
        1,
        79
    )
    .sum()
)


ocq180_80 = int(
    df["OCQ180"]
    .eq(80)
    .sum()
)


ocq180_77777 = int(
    df["OCQ180"]
    .eq(77777)
    .sum()
)


ocq180_99999 = int(
    df["OCQ180"]
    .eq(99999)
    .sum()
)


ocq180_missing = int(
    df["OCQ180"]
    .isna()
    .sum()
)


if ocq180_range != 3326:
    raise ValueError(
        "OCQ180 1-79 count differs."
    )


if ocq180_80 != 58:
    raise ValueError(
        "OCQ180 80+ count differs."
    )


if ocq180_77777 != 1:
    raise ValueError(
        "OCQ180 refused count differs."
    )


if ocq180_99999 != 1:
    raise ValueError(
        "OCQ180 don't-know count differs."
    )


if ocq180_missing != 2941:
    raise ValueError(
        "OCQ180 missing count differs."
    )


print(
    "\nPASS: OCQ180 matches CDC."
)


# ============================================================
# 17. OCD150 -> OCQ180 ROUTING
# ============================================================

#
# OCD150=1 means currently working.
#
# Exact attachment:
#
#     OCD150=1        = 3,386
#     OCQ180 populated = 3,386
#

ocq180_expected = (
    df["OCD150"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


ocq180_present = (
    df["OCQ180"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ocq180_routing_mismatch = int(
    np.sum(
        ocq180_expected
        !=
        ocq180_present
    )
)


if ocq180_routing_mismatch != 0:

    raise ValueError(
        "OCQ180 availability does not match OCD150=1."
    )


print(
    "PASS: OCD150/OCQ180 routing validated."
)


# ============================================================
# 18. CDC CHECK - OCQ210
# ============================================================

check_frequency(
    "OCQ210",
    {
        1: 383,
        2: 728,
        7: 1,
        9: 3
    },
    5212
)


# ============================================================
# 19. OCQ210 ROUTING CHECK
# ============================================================

#
# OCQ210 is asked for:
#
#     OCD150 = 2
#
# OR
#
# current workers reporting <35 hours,
# plus the small RF/DK OCQ180 paths.
#

ocq210_expected = (
    (
        df["OCD150"].eq(2)
        |
        (
            df["OCD150"].eq(1)
            &
            (
                df["OCQ180"]
                .between(1, 34)
                |
                df["OCQ180"]
                .isin(
                    [77777, 99999]
                )
            )
        )
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


ocq210_present = (
    df["OCQ210"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ocq210_mismatch = int(
    np.sum(
        ocq210_expected
        !=
        ocq210_present
    )
)


if ocq210_mismatch != 0:

    raise ValueError(
        "OCQ210 routing differs from attachment."
    )


print(
    "PASS: OCQ210 routing validated."
)


# ============================================================
# 20. CDC CHECK - OCQ260
# ============================================================

check_frequency(
    "OCQ260",
    {
        1: 2571,
        2: 89,
        3: 239,
        4: 184,
        5: 372,
        6: 24,
        77: 2,
        99: 38
    },
    2808
)


# ============================================================
# 21. CURRENT-JOB FIELD ROUTING
# ============================================================

#
# OCD150:
#
#     1 = Working
#     2 = Has job but not at work
#
# Together:
#
#     3,519 records
#
#
# OCQ260, OCD270, OCQ600 are populated exactly
# for those 3,519 current-job participants.
#

current_job_expected = (
    df["OCD150"]
    .isin(
        [1, 2]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


if int(
    current_job_expected.sum()
) != 3519:

    raise ValueError(
        "Unexpected current-job participant count."
    )


for col in [
    "OCQ260",
    "OCD270",
    "OCQ600"
]:

    present = (
        df[col]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )


    mismatch = int(
        np.sum(
            current_job_expected
            !=
            present
        )
    )


    if mismatch != 0:

        raise ValueError(
            f"{col} current-job routing differs."
        )


print(
    "PASS: Current-job routing validated "
    "for OCQ260/OCD270/OCQ600."
)


# ============================================================
# 22. CDC CHECK - OCD270
# ============================================================

#
# Months in current main job:
#
#     1-468   = 3,391
#     0       =   105   Less than 1 month
#     480     =    20   480 months or more
#     77777   =     3
#
# Missing = 2,808
#

ocd270_range = int(
    df["OCD270"]
    .between(
        1,
        468
    )
    .sum()
)


ocd270_zero = int(
    df["OCD270"]
    .eq(0)
    .sum()
)


ocd270_480 = int(
    df["OCD270"]
    .eq(480)
    .sum()
)


ocd270_77777 = int(
    df["OCD270"]
    .eq(77777)
    .sum()
)


ocd270_missing = int(
    df["OCD270"]
    .isna()
    .sum()
)


if ocd270_range != 3391:
    raise ValueError(
        "OCD270 1-468 count differs."
    )


if ocd270_zero != 105:
    raise ValueError(
        "OCD270 zero count differs."
    )


if ocd270_480 != 20:
    raise ValueError(
        "OCD270 480+ count differs."
    )


if ocd270_77777 != 3:
    raise ValueError(
        "OCD270 refused count differs."
    )


if ocd270_missing != 2808:
    raise ValueError(
        "OCD270 missing count differs."
    )


print(
    "PASS: OCD270 matches CDC."
)


# ============================================================
# 23. CDC CHECK - OCQ600
# ============================================================

check_frequency(
    "OCQ600",
    {
        1: 196,
        2: 60,
        3: 74,
        4: 208,
        5: 2510,
        6: 469,
        77: 1,
        99: 1
    },
    2808
)


# ============================================================
# 24. CDC CHECK - OCQ610
# ============================================================

#
# Days/month exposed to loud noise:
#
# CDC:
#
#     0-30 = 3,509
#     77   =     1
#     99   =     4
#
# Missing = 2,813
#
#
# Direct attachment after repair:
#
#     zero   = 2,495
#     1-30   = 1,014
#

ocq610_zero = int(
    df["OCQ610"]
    .eq(0)
    .sum()
)


ocq610_positive = int(
    df["OCQ610"]
    .between(
        1,
        30
    )
    .sum()
)


ocq610_total_range = int(
    df["OCQ610"]
    .between(
        0,
        30
    )
    .sum()
)


ocq610_77 = int(
    df["OCQ610"]
    .eq(77)
    .sum()
)


ocq610_99 = int(
    df["OCQ610"]
    .eq(99)
    .sum()
)


ocq610_missing = int(
    df["OCQ610"]
    .isna()
    .sum()
)


print(
    "\n--- OCQ610 ---"
)

print(
    "Zero:",
    f"{ocq610_zero:,}"
)

print(
    "1-30:",
    f"{ocq610_positive:,}"
)

print(
    "0-30 total:",
    f"{ocq610_total_range:,}"
)


if ocq610_zero != 2495:
    raise ValueError(
        "OCQ610 zero count differs."
    )


if ocq610_positive != 1014:
    raise ValueError(
        "OCQ610 positive range count differs."
    )


if ocq610_total_range != 3509:
    raise ValueError(
        "OCQ610 0-30 range count differs."
    )


if ocq610_77 != 1:
    raise ValueError(
        "OCQ610 refused count differs."
    )


if ocq610_99 != 4:
    raise ValueError(
        "OCQ610 don't-know count differs."
    )


if ocq610_missing != 2813:
    raise ValueError(
        "OCQ610 missing count differs."
    )


print(
    "PASS: OCQ610 matches CDC and attachment."
)


# ============================================================
# 25. CDC CHECK - OCQ630
# ============================================================

#
# Hours/day exposed to loud noise:
#
#     1-20 = 1,018
#     77   =     1
#
# Missing = 5,308
#

ocq630_range = int(
    df["OCQ630"]
    .between(
        1,
        20
    )
    .sum()
)


ocq630_77 = int(
    df["OCQ630"]
    .eq(77)
    .sum()
)


ocq630_missing = int(
    df["OCQ630"]
    .isna()
    .sum()
)


if ocq630_range != 1018:
    raise ValueError(
        "OCQ630 1-20 count differs."
    )


if ocq630_77 != 1:
    raise ValueError(
        "OCQ630 refused count differs."
    )


if ocq630_missing != 5308:
    raise ValueError(
        "OCQ630 missing count differs."
    )


print(
    "PASS: OCQ630 matches CDC."
)


# ============================================================
# 26. CDC CHECK - OCQ640
# ============================================================

#
# Days/month exposed to VERY loud noise:
#
# CDC:
#
#     0-30 = 3,508
#     77   =     1
#     99   =     3
#
# Missing = 2,815
#
#
# Exact attachment zero count:
#
#     2,918
#

ocq640_zero = int(
    df["OCQ640"]
    .eq(0)
    .sum()
)


ocq640_positive = int(
    df["OCQ640"]
    .between(
        1,
        30
    )
    .sum()
)


ocq640_total_range = int(
    df["OCQ640"]
    .between(
        0,
        30
    )
    .sum()
)


ocq640_77 = int(
    df["OCQ640"]
    .eq(77)
    .sum()
)


ocq640_99 = int(
    df["OCQ640"]
    .eq(99)
    .sum()
)


ocq640_missing = int(
    df["OCQ640"]
    .isna()
    .sum()
)


if ocq640_zero != 2918:
    raise ValueError(
        "OCQ640 zero count differs."
    )


if ocq640_positive != 590:
    raise ValueError(
        "OCQ640 positive range count differs."
    )


if ocq640_total_range != 3508:
    raise ValueError(
        "OCQ640 0-30 range count differs."
    )


if ocq640_77 != 1:
    raise ValueError(
        "OCQ640 refused count differs."
    )


if ocq640_99 != 3:
    raise ValueError(
        "OCQ640 don't-know count differs."
    )


if ocq640_missing != 2815:
    raise ValueError(
        "OCQ640 missing count differs."
    )


print(
    "PASS: OCQ640 matches CDC and attachment."
)


# ============================================================
# 27. CDC CHECK - OCQ660
# ============================================================

#
# Hours/day exposed to very loud noise:
#
#     1-20 = 593
#     77   = 1
#
# Missing = 5,733
#

ocq660_range = int(
    df["OCQ660"]
    .between(
        1,
        20
    )
    .sum()
)


ocq660_77 = int(
    df["OCQ660"]
    .eq(77)
    .sum()
)


ocq660_missing = int(
    df["OCQ660"]
    .isna()
    .sum()
)


if ocq660_range != 593:
    raise ValueError(
        "OCQ660 1-20 count differs."
    )


if ocq660_77 != 1:
    raise ValueError(
        "OCQ660 refused count differs."
    )


if ocq660_missing != 5733:
    raise ValueError(
        "OCQ660 missing count differs."
    )


print(
    "PASS: OCQ660 matches CDC."
)


# ============================================================
# 28. CDC CHECK - OCQ380
# ============================================================

check_frequency(
    "OCQ380",
    {
        1: 427,
        2: 315,
        3: 1001,
        4: 197,
        5: 21,
        6: 358,
        7: 218,
        77: 1,
        99: 1
    },
    3788
)


# ============================================================
# 29. OCD150 -> OCQ380 ROUTING
# ============================================================

#
# OCD150 = 4:
#
#     Not working at a job or business.
#
# Exact:
#
#     OCD150=4 = 2,539
#     OCQ380 nonmissing = 2,539
#

ocq380_expected = (
    df["OCD150"]
    .eq(4)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


ocq380_present = (
    df["OCQ380"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ocq380_mismatch = int(
    np.sum(
        ocq380_expected
        !=
        ocq380_present
    )
)


if ocq380_mismatch != 0:

    raise ValueError(
        "OCQ380 availability does not match OCD150=4."
    )


print(
    "PASS: OCD150/OCQ380 routing validated."
)


# ============================================================
# 30. CDC CHECK - OCD390G
# ============================================================

check_frequency(
    "OCD390G",
    {
        1: 3589,
        2: 2286,
        3: 59,
        4: 373,
        7: 9,
        9: 9
    },
    2
)


# ============================================================
# 31. CDC CHECK - OCD395
# ============================================================

#
# Duration of longest job:
#
#     1-588 = 3,584
#     0     =    19  Less than 1 month
#     600   =    33  600 months or more
#     99999 =    12
#
# Missing = 2,679
#

ocd395_range = int(
    df["OCD395"]
    .between(
        1,
        588
    )
    .sum()
)


ocd395_zero = int(
    df["OCD395"]
    .eq(0)
    .sum()
)


ocd395_600 = int(
    df["OCD395"]
    .eq(600)
    .sum()
)


ocd395_99999 = int(
    df["OCD395"]
    .eq(99999)
    .sum()
)


ocd395_77777 = int(
    df["OCD395"]
    .eq(77777)
    .sum()
)


ocd395_missing = int(
    df["OCD395"]
    .isna()
    .sum()
)


if ocd395_range != 3584:
    raise ValueError(
        "OCD395 1-588 count differs."
    )


if ocd395_zero != 19:
    raise ValueError(
        "OCD395 zero count differs."
    )


if ocd395_600 != 33:
    raise ValueError(
        "OCD395 600+ count differs."
    )


if ocd395_99999 != 12:
    raise ValueError(
        "OCD395 don't-know count differs."
    )


if ocd395_77777 != 0:
    raise ValueError(
        "Unexpected OCD395 refused values."
    )


if ocd395_missing != 2679:
    raise ValueError(
        "OCD395 missing count differs."
    )


print(
    "PASS: OCD395 matches CDC."
)


# ============================================================
# 32. OCD390G -> OCD395 ROUTING CHECK
# ============================================================

#
# OCD395 is populated when OCD390G is:
#
#     1 = Enter longest occupation
#     3 = Armed forces
#
#
# Exact:
#
#     OCD390G in {1,3} = 3,648
#     OCD395 populated = 3,648
#

ocd395_expected = (
    df["OCD390G"]
    .isin(
        [1, 3]
    )
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


ocd395_present = (
    df["OCD395"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


ocd395_routing_mismatch = int(
    np.sum(
        ocd395_expected
        !=
        ocd395_present
    )
)


if ocd395_routing_mismatch != 0:

    raise ValueError(
        "OCD395 availability differs from OCD390G."
    )


print(
    "PASS: OCD390G/OCD395 routing validated."
)


# ============================================================
# 33. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Original ordinary zero count was 0.
#
# Therefore after repair the ONLY zeros should be:
#
#     OCD270 =   105
#     OCQ610 = 2,495
#     OCQ640 = 2,918
#     OCD395 =    19
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 5537:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 5,537 validated restorations."
)


# ============================================================
# 34. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# ONLY the validated tiny -> zero repairs may differ
# from source_df.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 14 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 35. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# OCQ_I has no genuine decimal fields, but we keep
# the same safe export approach used throughout NHANES.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 36. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 37. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 14 OCQ_I variables."
)


# ============================================================
# 38. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 39. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 40. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 41. RAW CSV .0 CHECK
# ============================================================

#
# OCQ_I contains no genuine decimal variables.
#
# Therefore checking all 14 variables for unwanted
# ".0" is safe for THIS file.
#

print(
    "\n--- RAW CSV .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any OCQ_I variable."
)


# ============================================================
# 42. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "the 5,537 validated zeros."
)


# ============================================================
# 43. FINAL TOP-CODE CHECK
# ============================================================

#
# Important 2015-16 disclosure-control top-codes:
#
# OCQ180:
#     80 = 80 hours or more
#
# OCD270:
#     480 = 480 months or more
#
# OCD395:
#     600 = 600 months or more
#

final_top_codes = {
    "OCQ180_80": int(
        pd.to_numeric(
            raw_csv_df["OCQ180"],
            errors="coerce"
        )
        .eq(80)
        .sum()
    ),

    "OCD270_480": int(
        pd.to_numeric(
            raw_csv_df["OCD270"],
            errors="coerce"
        )
        .eq(480)
        .sum()
    ),

    "OCD395_600": int(
        pd.to_numeric(
            raw_csv_df["OCD395"],
            errors="coerce"
        )
        .eq(600)
        .sum()
    )
}


print(
    "\n--- FINAL TOP-CODE CHECK ---"
)

print(
    final_top_codes
)


EXPECTED_TOP_CODES = {
    "OCQ180_80": 58,
    "OCD270_480": 20,
    "OCD395_600": 33
}


if final_top_codes != EXPECTED_TOP_CODES:

    raise ValueError(
        "Final top-code distribution changed."
    )


print(
    "PASS: 2015-16 top-codes preserved."
)


# ============================================================
# 44. FINAL ROUTING CHECKS
# ============================================================

#
# Re-check strongest routing relationships after export.
#

final_ocd150 = pd.to_numeric(
    raw_csv_df["OCD150"],
    errors="coerce"
)


final_ocq180_present = (
    raw_csv_df["OCQ180"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


expected_ocq180 = (
    final_ocd150
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_ocq180_mismatch = int(
    np.sum(
        expected_ocq180
        !=
        final_ocq180_present
    )
)


final_ocq380_present = (
    raw_csv_df["OCQ380"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


expected_ocq380 = (
    final_ocd150
    .eq(4)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_ocq380_mismatch = int(
    np.sum(
        expected_ocq380
        !=
        final_ocq380_present
    )
)


final_ocd390g = pd.to_numeric(
    raw_csv_df["OCD390G"],
    errors="coerce"
)


expected_ocd395 = (
    final_ocd390g
    .isin([1, 3])
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


final_ocd395_present = (
    raw_csv_df["OCD395"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_ocd395_mismatch = int(
    np.sum(
        expected_ocd395
        !=
        final_ocd395_present
    )
)


print(
    "\n--- FINAL ROUTING CHECKS ---"
)

print(
    "OCD150/OCQ180 mismatches:",
    final_ocq180_mismatch
)

print(
    "OCD150/OCQ380 mismatches:",
    final_ocq380_mismatch
)

print(
    "OCD390G/OCD395 mismatches:",
    final_ocd395_mismatch
)


if final_ocq180_mismatch != 0:

    raise ValueError(
        "Final OCD150/OCQ180 routing changed."
    )


if final_ocq380_mismatch != 0:

    raise ValueError(
        "Final OCD150/OCQ380 routing changed."
    )


if final_ocd395_mismatch != 0:

    raise ValueError(
        "Final OCD390G/OCD395 routing changed."
    )


print(
    "PASS: Final routing relationships preserved."
)


# ============================================================
# 45. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 46. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 47. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "OCQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - Occupation"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Tiny artifact variables:",
    list(
        EXPECTED_TINY_COUNTS.keys()
    )
)


print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "OCD270 zeros:",
    ocd270_zero
)


print(
    "OCQ610 zeros:",
    f"{ocq610_zero:,}"
)


print(
    "OCQ640 zeros:",
    f"{ocq640_zero:,}"
)


print(
    "OCD395 zeros:",
    ocd395_zero
)


print(
    "Genuine decimal variables:",
    "NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "Current-job records:",
    int(
        current_job_expected.sum()
    )
)


print(
    "OCQ180 80+ top-code records:",
    ocq180_80
)


print(
    "OCD270 480+ top-code records:",
    ocd270_480
)


print(
    "OCD395 600+ top-code records:",
    ocd395_600
)


print(
    "OCD150/OCQ180 routing mismatches:",
    ocq180_routing_mismatch
)


print(
    "OCD150/OCQ380 routing mismatches:",
    ocq380_mismatch
)


print(
    "OCD390G/OCD395 routing mismatches:",
    ocd395_routing_mismatch
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "OCQ_I contains no genuine decimal variables."
)


print(
    "Only the 5,537 validated XPORT tiny values "
    "are changed to real numeric zero."
)


print(
    "All 14 fields can then safely be exported "
    "as whole-number IDs/codes/counts."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "variables remain exactly AS IS."
)

NHANES 2015-2016 OCQ_I XPT -> CSV
Rows: 6,327
Columns: 14
PASS: Structure = 6,327 rows x 14 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: []
String fields: []
Whole-number/code fields: ['SEQN', 'OCD150', 'OCQ180', 'OCQ210', 'OCQ260', 'OCD270', 'OCQ600', 'OCQ610', 'OCQ630', 'OCQ640', 'OCQ660', 'OCQ380', 'OCD390G', 'OCD395']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 5,537
Variables containing tiny artifacts: 4

Tiny values by variable:
OCQ640    2918
OCQ610    2495
OCD270     105
OCD395      19

PASS: Exact 5,537-value tiny-zero pattern across 4 variables validated.

--- VALIDATED ZERO RESTORATION ---
OCD270: 105 restored to 0
OCQ610: 2,495 restored to 0
OCQ640: 2,918 restored to 0
OCD395: 19 restored to 0

PASS: 5,537 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
OCD150: 0 fractional observations
OCQ180: 0 fra